In [1]:
from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI
from typing import TypedDict
from dotenv import load_dotenv

In [ ]:
load_dotenv()

model = ChatOpenAI(model="gpt-4o-mini")

In [ ]:
class BlogState(TypedDict):
    title: str
    outline: str
    content: str
    evaluation: str
    score: int

In [2]:

def create_outline(state: BlogState) -> BlogState:

    # Fetch title
    title = state["title"]

    # Call LLM to generate outline
    prompt = f"Generate a detailed outline for a blog on the topic - {title}"
    outline = model.invoke(prompt).content

    # Update state
    state["outline"] = outline

    return state


NameError: name 'BlogState' is not defined

In [ ]:

def create_blog(state: BlogState) -> BlogState:

    # Fetch title and outline
    title = state["title"]
    outline = state["outline"]

    # Form prompt
    prompt = f"Write a detailed blog on the title - {title} using the following outline:\n{outline}"

    # Call LLM to generate blog
    content = model.invoke(prompt).content

    # Update state
    state["content"] = content

    return state


In [ ]:

# def evaluate_blog(state: BlogState) -> BlogState:

#     # Fetch the generated blog
#     title = state["title"]
#     content = state["content"]

#     # Create an evaluation prompt
#     prompt = f"""
#     Evaluate the following blog.

#     Blog title: {title}

#     Blog content:
#     {content}

#     Evaluate the blog based on:
#     1. Relevance to the title
#     2. Structure and organization
#     3. Clarity and readability
#     4. Depth and usefulness
#     5. Grammar and writing quality

#     Give an overall score out of 10.

#     Return your response in this format:
#     Score: X/10
#     Evaluation: Your detailed feedback
#     """

#     # Ask the LLM to evaluate the blog
#     result = model.invoke(prompt).content

#     # Update the state
#     state["evaluation"] = result

#     return state


In [ ]:

import re

def evaluate_blog(state: BlogState) -> BlogState:

    title = state["title"]
    content = state["content"]

    prompt = f"""
    Evaluate this blog on a scale of 1 to 10.

    Title: {title}

    Blog:
    {content}

    Consider relevance, structure, clarity, depth, and grammar.

    Return exactly this format:
    Score: 8
    Feedback: Explain the strengths and weaknesses.
    """

    result = model.invoke(prompt).content

    # Extract the score
    match = re.search(r"Score:\s*(\d{1,2})", result)

    if not match:
        raise ValueError("The LLM did not return a valid score.")

    score = int(match.group(1))

    if not 1 <= score <= 10:
        raise ValueError("The score must be between 1 and 10.")

    # Update the state
    state["score"] = score
    state["evaluation"] = result

    return state


In [ ]:
#graph
graph = StateGraph(BlogState)

#node
graph.add_node('create_outline', create_outline)
graph.add_node('create_blog', create_blog)
graph.add_node("evaluate_blog", evaluate_blog)

#edge
graph.add_edge(START, "create_outline")
graph.add_edge("create_outline", "create_blog")
graph.add_edge("create_blog", "evaluate_blog")
graph.add_edge("evaluate_blog", END)
#COMPILE

workflow = graph.compile()



NameError: name 'BlogState' is not defined

In [ ]:

initial_state = {
    "title": "The Future of Artificial Intelligence",
    "outline": "",
    "content": "",
    "evaluation": "",
    "score": 0
}


output = workflow.invoke(initial_state)

print("BLOG OUTLINE:\n")
print(output["outline"])

print("\nGENERATED BLOG:\n")
print(output["content"])

print("\nBLOG SCORE:", output["score"], "/10")

print("\nEVALUATION:\n")
print(output["evaluation"])
print(output["content"])
